# Real-time Bandpass Filter - Streaming vs Offline

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

This notebook demonstrates **real-time bandpass filtering** with streaming data. We design a Butterworth bandpass filter (0.5-30 Hz, order 4) and apply it using `scipy.signal.lfilter` with state maintenance between chunks. We compare this to the offline `filtfilt` approach, which requires the entire signal.

## What this notebook does

1. Loads the P4 channel from the local EEG dataset
2. Designs a Butterworth bandpass filter (0.5-30 Hz, order 4)
3. Processes the signal in chunks of 50 samples using `lfilter` with `zi` state maintenance
4. Computes the offline filtered version using `filtfilt` for comparison
5. Plots both results to show the difference

## What you should expect to see

- The top plot shows the **original raw signal** (first 5000 samples) in blue
- The bottom plot shows the **real-time filtered signal** in green (using lfilter with state)
- The **offline filtered signal** is overlaid in red dashed for comparison
- The real-time filter has a slight transient at the beginning but converges to match the offline version

## Key parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| FS | 200 Hz | Sampling rate |
| LOWCUT | 0.5 Hz | Bandpass low cutoff |
| HIGHCUT | 30.0 Hz | Bandpass high cutoff |
| ORDER | 4 | Butterworth filter order |
| CHUNK_SIZE | 50 | Samples per processing chunk |

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone repo and download data

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel **P4** (parietal region).

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. Apply the analysis

We design a Butterworth bandpass filter (0.5-30 Hz, order 4) and apply it in two ways:
1. **Real-time** using `lfilter` with `zi` state maintenance between chunks of 50 samples
2. **Offline** using `filtfilt` for zero-phase filtering (requires the whole signal)

In [ ]:
from scipy import signal

LOWCUT = 0.5
HIGHCUT = 30.0
ORDER = 4
CHUNK_SIZE = 50
N_PLOT = 5000

n_samples = min(N_PLOT, len(channel_data))
signal_plot = channel_data[:n_samples]

nyq = 0.5 * fs
b, a = signal.butter(ORDER, [LOWCUT / nyq, HIGHCUT / nyq], btype='band', analog=False)

zi = signal.lfilter_zi(b, a)
state = zi * signal_plot[0]
realtime_filtered = np.zeros(n_samples)

for start in range(0, n_samples, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n_samples)
    chunk = signal_plot[start:end]
    filtered_chunk, state = signal.lfilter(b, a, chunk, zi=state)
    realtime_filtered[start:end] = filtered_chunk

offline_filtered = signal.filtfilt(b, a, signal_plot)

print(f'Bandpass: {LOWCUT}-{HIGHCUT} Hz, order {ORDER}')
print(f'Real-time filtered {n_samples} samples in chunks of {CHUNK_SIZE}')
print(f'Offline filtered using filtfilt (zero-phase)')

## 5. Interactive plot

**What to look for:**
- The top plot shows the original raw signal with all its noise and drift
- The bottom plot shows the real-time filtered signal (green) and the offline filtered signal (red dashed)
- The real-time filter has a **slight transient** at the very beginning (first few samples) due to filter initialization
- After the transient, the two signals **converge** and look nearly identical
- The offline `filtfilt` has zero phase delay, while `lfilter` introduces a small phase shift

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

x = np.arange(n_samples)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original Signal (P4)',
                                    f'Bandpass Filtered ({LOWCUT}-{HIGHCUT} Hz)'))

fig.add_trace(go.Scatter(x=x, y=signal_plot, name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=x, y=realtime_filtered, name='Real-time (lfilter)',
                         line=dict(color='green', width=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=offline_filtered, name='Offline (filtfilt)',
                         line=dict(color='red', width=0.5, dash='dash')), row=2, col=1)

fig.update_layout(height=600, title_text='Real-time Bandpass Filter - Streaming vs Offline',
                  xaxis2_title='Sample index', yaxis_title='Amplitude (uV)',
                  yaxis2_title='Amplitude (uV)')
fig.show()

## What did we learn?

- **`lfilter`** processes data sequentially and maintains **filter state** (`zi`) between chunks
- **`filtfilt`** applies the filter forward and backward for zero-phase, but requires the **entire signal**
- The `zi` parameter in `lfilter` carries the filter's internal state across chunk boundaries
- A slight **transient** appears at the start of real-time filtering, but it converges quickly
- Real-time filtering introduces a small **phase delay** that offline filtering avoids
- This approach is essential for **online BCI systems** that must process data as it arrives